In [2]:
import os
import torch
import torch.nn as nn
from transformers import Wav2Vec2PreTrainedModel, Wav2Vec2Model, Wav2Vec2Config, AutoProcessor

# ==============================================================================
# 1. 定义模型架构 (保持与训练代码完全一致)
# ==============================================================================
class MultiScaleAttentivePooling(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(hidden_size, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        attn_weights = torch.softmax(self.attention(x), dim=1)
        mu = torch.sum(attn_weights * x, dim=1)
        delta = x - mu.unsqueeze(1)
        var = torch.sum(attn_weights * (delta ** 2), dim=1)
        std = torch.sqrt(torch.clamp(var, min=1e-9))
        return torch.cat([mu, std], dim=-1)

class VoxSentinelForEmotion(Wav2Vec2PreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.wav2vec2 = Wav2Vec2Model(config)
        self.pooling = MultiScaleAttentivePooling(config.hidden_size)
        self.classifier = nn.Sequential(
            nn.Linear(config.hidden_size * 2, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, config.num_labels)
        )
        self.post_init()

    def forward(self, input_values, attention_mask=None):
        outputs = self.wav2vec2(input_values, attention_mask=attention_mask)
        pooled_output = self.pooling(outputs.last_hidden_state)
        logits = self.classifier(pooled_output)
        return logits

# ==============================================================================
# 2. 权重加载与自动键名转换
# ==============================================================================
checkpoint_path = "./model/best_emotion_model.pth" # 请确保这是你情绪模型的路径
export_dir = "./VoxSentinel_Emotion_Release"

# 情绪标签定义
label2id = {'angry': 0, 'disgust': 1, 'fear': 2, 'happy': 3, 'neutral': 4, 'sad': 5, 'surprise': 6}
id2label = {v: k for k, v in label2id.items()}

print("🏗️ Initializing structure and mapping weights...")

if not os.path.exists(checkpoint_path):
    print(f"❌ Error: Checkpoint not found at {checkpoint_path}")
else:
    # 1. 初始化配置
    config = Wav2Vec2Config.from_pretrained(
        "facebook/wav2vec2-base", 
        num_labels=len(label2id),
        label2id=label2id,
        id2label=id2label
    )
    
    # 2. 实例化模型
    hf_model = VoxSentinelForEmotion(config)
    
    # 3. 加载原始权重字典
    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    
    # 如果你的 .pth 是用 torch.save({'model_state_dict': model.state_dict()}) 保存的
    raw_state_dict = checkpoint['model_state_dict'] if 'model_state_dict' in checkpoint else checkpoint
    
    # 🌟 核心修复逻辑：重命名键名以匹配 HF 类结构
    new_state_dict = {}
    for k, v in raw_state_dict.items():
        if k.startswith("backbone."):
            # 将 backbone. 替换为 wav2vec2.
            new_key = k.replace("backbone.", "wav2vec2.")
            new_state_dict[new_key] = v
        else:
            # pooling. 和 classifier. 的键名通常不需要修改
            new_state_dict[k] = v
            
    # 4. 强制加载（strict=True 确保没有遗漏）
    try:
        hf_model.load_state_dict(new_state_dict, strict=True)
        print("✅ Weights mapped successfully!")
    except RuntimeError as e:
        print(f"⚠️ Strict load failed, attempting non-strict load. Error: {e}")
        hf_model.load_state_dict(new_state_dict, strict=False)

    # 5. 保存打包文件
    hf_model.save_pretrained(export_dir)
    processor = AutoProcessor.from_pretrained("facebook/wav2vec2-base")
    processor.save_pretrained(export_dir)
    
    print(f"\n✨ SUCCESS! Bundle saved to: {export_dir}")
    print(f"📦 You can now upload the contents of '{export_dir}' to Hugging Face.")

🏗️ Initializing structure and mapping weights...
✅ Weights mapped successfully!


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✨ SUCCESS! Bundle saved to: ./VoxSentinel_Emotion_Release
📦 You can now upload the contents of './VoxSentinel_Emotion_Release' to Hugging Face.


In [6]:
import torch
import librosa
from transformers import AutoProcessor

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_path = "./VoxSentinel_Emotion_Release"

print(f"🧪 Testing packed model from {model_path}...")

# 加载模型和处理器
model = VoxSentinelForEmotion.from_pretrained(model_path).to(device)
processor = AutoProcessor.from_pretrained(model_path)
model.eval()

def predict_emotion(audio_path):
    speech, _ = librosa.load(audio_path, sr=16000)
    inputs = processor(speech, return_tensors="pt", sampling_rate=16000).input_values.to(device)
    
    with torch.no_grad():
        logits = model(inputs)
        pred_idx = torch.argmax(logits, dim=1).item()
    
    return model.config.id2label[pred_idx]

# 测试一个样本（你可以换成你自己的音频路径）
result = predict_emotion("./Raw_Data/Tess/OAF_Pleasant_surprise/OAF_came_ps.wav")
print(f"🔍 Prediction: {result}")

🧪 Testing packed model from ./VoxSentinel_Emotion_Release...


Loading weights:   0%|          | 0/226 [00:00<?, ?it/s]

🔍 Prediction: surprise


In [7]:
import torch
import torch.nn as nn
import librosa
from transformers import Wav2Vec2PreTrainedModel, Wav2Vec2Model, AutoProcessor

# 1. Define the Architecture (Required for loading custom layers)
class MultiScaleAttentivePooling(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(hidden_size, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )
    def forward(self, x):
        attn_weights = torch.softmax(self.attention(x), dim=1)
        mu = torch.sum(attn_weights * x, dim=1)
        delta = x - mu.unsqueeze(1)
        var = torch.sum(attn_weights * (delta ** 2), dim=1)
        std = torch.sqrt(torch.clamp(var, min=1e-9))
        return torch.cat([mu, std], dim=-1)

class VoxSentinelForEmotion(Wav2Vec2PreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.wav2vec2 = Wav2Vec2Model(config)
        self.pooling = MultiScaleAttentivePooling(config.hidden_size)
        self.classifier = nn.Sequential(
            nn.Linear(config.hidden_size * 2, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, config.num_labels)
        )
        self.post_init()

    def forward(self, input_values, attention_mask=None):
        outputs = self.wav2vec2(input_values, attention_mask=attention_mask)
        pooled_output = self.pooling(outputs.last_hidden_state)
        logits = self.classifier(pooled_output)
        return logits

# 2. Load Model and Processor
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
repo_id = "JesseHuang922/VoxSentinel-Emotion-Base"

model = VoxSentinelForEmotion.from_pretrained(repo_id).to(device)
processor = AutoProcessor.from_pretrained(repo_id)
model.eval()

# 3. Predict Function
def predict(audio_path):
    # Load and resample to 16kHz
    speech, _ = librosa.load(audio_path, sr=16000)
    inputs = processor(speech, return_tensors="pt", sampling_rate=16000).input_values.to(device)
    
    with torch.no_grad():
        logits = model(inputs)
        pred_idx = torch.argmax(logits, dim=1).item()
    
    return model.config.id2label[pred_idx]

# Example Usage
result = predict_emotion("./Raw_Data/Tess/OAF_Pleasant_surprise/OAF_came_ps.wav")
print(f"🔍 Prediction: {result}")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/382M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/226 [00:00<?, ?it/s]

processor_config.json:   0%|          | 0.00/300 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json:   0%|          | 0.00/358 [00:00<?, ?B/s]

🔍 Prediction: surprise
